<div class="alert alert-info">

<h3>Задание (выполнять в отдельном файле)</h3>
<p></p>
Вспомним задачу от «Сибур Диджитал» (train.parquet). Целевые переменные target0 и target1 имели бимодальное распределение. Мы разбивали выборку по признаку feature4 на две, при этом целевые переменные каждой из полученных выборок лучше описывалась гамма-распределением.
    
<ol>
    <li>Подгрузите всю таблицу в память через pd.read_parquet. Разбейте выборку на две части по признаку feature4.</li>
    <li>Преобразуйте каждую из выборок в Dataset, после чего разбейте их на обучение и контроль в соотношении 1:1.</li>
    <li>Реализуйте архитектуру нейронной сети из предыдущего задания, внося необходимые изменения для входного и выходного слоёв. В этой задаче необходимо оценить две целевые переменные. Соответственно, выходной слой должен состоять из двух нейронов.</li>
    <li>Поскольку целевые переменные лучше описывались гамма-распределением, используйте соответствующую функцию потерь, представленную ниже. Обучайте нейронную сеть 120 эпох по мини-батчам размером 64. Для оптимизации используйте метод Adam с шагом обучения 0.001.</li>
    <li>Получите предсказания на контрольной выборке и оцените их с помощью метрики MAPE.</li>
    
</ol>
</div>

$$L(\hat{y}, y)=\frac{y}{\hat{y}} + log{\hat{y}}$$

In [116]:
def GammaLoss(output, target):
    loss = target / output + torch.log(output)
    return loss.mean()

In [117]:
import pandas as pd

In [118]:
df = pd.read_parquet("data/train.parquet")

In [119]:
df_1 = df[df['feature4'] == 'gas1'].reset_index(drop=True)
df_2 = df[df['feature4'] == 'gas2'].reset_index(drop=True)

In [120]:
class GasDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [121]:
feature_cols = [col for col in df.columns if col.startswith('feature') and col != 'feature4']
target_cols = ['target0', 'target1']

In [122]:
X1 = df_1[feature_cols].values
y1 = df_1[target_cols].values
dataset1 = GasDataset(X1, y1)

X2 = df_2[feature_cols].values
y2 = df_2[target_cols].values
dataset2 = GasDataset(X2, y2)

In [123]:
train_size = 0.5
test_size = 0.5

train_dataset1, test_dataset1 = random_split(dataset1, [train_size, test_size])
train_dataset2, test_dataset2 = random_split(dataset2, [train_size, test_size])

In [124]:
batch_size = 64

train_loader1 = DataLoader(train_dataset1, batch_size=batch_size, shuffle=True)
test_loader1 = DataLoader(test_dataset1, batch_size=batch_size, shuffle=False)

train_loader2 = DataLoader(train_dataset2, batch_size=batch_size, shuffle=True)
test_loader2 = DataLoader(test_dataset2, batch_size=batch_size, shuffle=False)

In [125]:
class GasNetwork(nn.Module):
    def __init__(self, input_size):
        super(GasNetwork, self).__init__()
        self.input_layer = nn.Linear(in_features=input_size, out_features=10)
        self.hidden_layer = nn.Linear(in_features=10, out_features=10)
        self.output_layer = nn.Linear(in_features=10, out_features=2)
        self.activation = nn.ReLU()
        
    def forward(self, x):
        x = self.activation(self.input_layer(x))
        x = self.activation(self.hidden_layer(x))
        return self.output_layer(x)

In [126]:
input_size = len(feature_cols)
model1 = GasNetwork(input_size)
model2 = GasNetwork(input_size)

In [127]:
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.001)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

n_epochs = 120

In [128]:
for epoch in range(n_epochs):
    losses = 0
    for x_batch, y_batch in train_loader1:
        optimizer1.zero_grad()
        pred = model1(x_batch)
        loss = GammaLoss(pred, y_batch)
        losses += loss.item() / len(train_loader1)
        loss.backward()
        optimizer1.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'epoch: {epoch + 1}  loss: {losses:.3f}')

epoch: 10  loss: 3.279
epoch: 20  loss: 3.273
epoch: 30  loss: 3.273
epoch: 40  loss: 3.273
epoch: 50  loss: 3.273
epoch: 60  loss: 3.273
epoch: 70  loss: 3.273
epoch: 80  loss: 3.273
epoch: 90  loss: 3.273
epoch: 100  loss: 3.273
epoch: 110  loss: 3.273
epoch: 120  loss: 3.273


In [129]:
for epoch in range(n_epochs):
    losses = 0
    for x_batch, y_batch in train_loader2:
        optimizer2.zero_grad()
        pred = model2(x_batch)
        loss = GammaLoss(pred, y_batch)
        losses += loss.item() / len(train_loader2)
        loss.backward()
        optimizer2.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'epoch: {epoch + 1}  loss: {losses:.3f}')

epoch: 10  loss: 5.103
epoch: 20  loss: 5.101
epoch: 30  loss: 5.098
epoch: 40  loss: 5.096
epoch: 50  loss: 5.090
epoch: 60  loss: 5.089
epoch: 70  loss: 5.089
epoch: 80  loss: 5.089
epoch: 90  loss: 5.089
epoch: 100  loss: 5.089
epoch: 110  loss: 5.089
epoch: 120  loss: 5.089


In [130]:
def calculate_mape(model, loader):
    model.eval()
    total_mape = 0
    count = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            pred = model(x_batch)
            mape = torch.mean(torch.abs((y_batch - pred) / y_batch))
            total_mape += mape.item() * len(y_batch)
            count += len(y_batch)
    return total_mape / count

In [131]:
mape1 = calculate_mape(model1, test_loader1)
mape2 = calculate_mape(model2, test_loader2)

In [133]:
print(f"Первое разделенеи (gas1): {mape1:.2f}")
print(f"Второе разделенеи (gas2): {mape2:.2f}")

Первое разделенеи (gas1): 0.02
Второе разделенеи (gas2): 0.01
